# BIOM5350 — Prelab: Version Control with git and GitHub

By the end of this prelab you will:

1. Have a working **GitHub account**, connected to your CoCalc project by SSH.
2. Be able to explain what **version control** is and why it matters for reproducible science.
3. Know the difference between **git** (the tool) and **GitHub** (the website).
4. Understand the **working directory → staging area → repository → remote** model.
5. Have made your first commits in a local repository.

## How to use this notebook

Most of the work in this prelab happens in a **terminal**, not in notebook cells. Open one alongside this notebook: in CoCalc, choose **View ▸ Terminal** and it opens in a split pane.

Terminal commands are shown in boxes like this, with `$>` standing for the terminal prompt. Type what follows `$>`, not the `$>` itself:

```
$> echo "type me in the terminal"
```

Cells beginning with `!` are **verification cells**. Run them with Shift+Enter to confirm that a step worked.

Cells labelled **ANSWER** are plain-text cells for your responses. Click into one, type your answer, and move on.

## Part A — Setup

### A1. Create a GitHub account

**Already have a GitHub account?** You are free to use it — skip to A2. The one thing to check: make sure 2FA is enabled (GitHub requires it; see item 4 below), and that you can still log in.

**If you do not have a GitHub account**, walk through the following:

1. Go to <https://github.com/signup>.
2. **Choose your username carefully.** You will likely use this account for years — on your CV, in publications, in job applications. Something like `jsmith` or `jane-smith-bio` is good; `xXsephiroth420Xx` is not. You can change it later, but links to your work will break.
3. Use an email you will keep after graduating (a personal email is fine; you can add your university email as a secondary one).
4. **Two-factor authentication (2FA).** GitHub requires it. The easiest path is an authenticator app on your phone (Google Authenticator, Microsoft Authenticator, Authy, 1Password, …). Save the recovery codes somewhere safe — if you lose your phone *and* your recovery codes, you lose the account.
5. *(Optional)* Apply for the **GitHub Student Developer Pack** at <https://education.github.com/pack> using your university email. It gives free access to a lot of tools, including GitHub Pro.

Record your GitHub username:

### A2. Tell git who you are

git stamps every commit with a name and email. Set them once in your CoCalc project. **Use the same email you used for GitHub**, so GitHub can link your commits to your account.

In the terminal, execute the following commands (substituting your own name and email):

```
$> git config --global user.name "Your Name"
$> git config --global user.email "you@example.com"
$> git config --global init.defaultBranch main
$> git config --global core.editor nano
```

The last two lines make new repositories use the branch name `main`, matching GitHub, and make git open the simple `nano` editor if it ever needs you to type a message.

Run the verification cell:

In [ ]:
!git config --global --list

You should see your `user.name`, `user.email`, `init.defaultbranch=main` and `core.editor=nano`.

### A3. Create an SSH key and add it to GitHub

When CoCalc talks to GitHub it needs to prove it is you. We use an **SSH key pair**: a *private* key that stays on CoCalc (never share it) and a *public* key you give to GitHub. Think of the public key as a padlock you hand out; only your private key opens it.

**Step 1 — generate the key.** In the terminal, execute the following command (substituting your email):

```
$> ssh-keygen -t ed25519 -C "you@example.com"
```

Press **Enter** three times to accept the default file location and no passphrase. (A passphrase is more secure but means typing it on every push; for this course, no passphrase is fine.)

**Step 2 — display the public key.** In the terminal, execute:

```
$> cat ~/.ssh/id_ed25519.pub
```

The output is a single line starting with `ssh-ed25519` and ending with your email. Select the whole line and copy it.

**Step 3 — give it to GitHub.**

1. On github.com, click your profile picture (top right) ▸ **Settings** ▸ **SSH and GPG keys** ▸ **New SSH key**.
2. Title: `CoCalc BIOM5350` (any name that reminds you where the key lives).
3. Key type: *Authentication Key*.
4. Paste the line you copied into the *Key* box and click **Add SSH key**.

**Step 4 — test it.** In the terminal, execute:

```
$> ssh -T git@github.com
```

The first time, it will ask `Are you sure you want to continue connecting (yes/no/[fingerprint])?` — type `yes`. You should then see:

```
Hi <your-username>! You've successfully authenticated, but GitHub does not provide shell access.
```

That message is success; the "does not provide shell access" part is normal. Now run the verification cell:

In [ ]:
# Verification: this should print "Hi <username>! You've successfully authenticated..."
!ssh -T -o StrictHostKeyChecking=accept-new git@github.com 2>&1 | head -n 3

If you see `Permission denied (publickey)` instead, the usual causes are: (a) you copied only part of the key line, (b) you pasted the *private* key (`id_ed25519`, without `.pub`) by mistake, or (c) you didn't click *Add SSH key*. Re-do Steps 2–3. If it still fails, post in the course Slack with the output of the cell above.

Paste the success line from the cell above:

## Part B — Concepts

### B1. Why version control?

Every bioinformatics project goes through the same story. You write `analysis.py`. It works. You change it, it breaks, and you can't remember what you changed. So you start saving copies: `analysis_v2.py`, `analysis_final.py`, `analysis_final_REAL.py`, `analysis_final_REAL_fixed.py`. Six months later a reviewer asks which version produced Figure 3, and nobody knows.

A **version control system** solves this by recording **snapshots** of your project over time. Each snapshot (a *commit*) records *what* changed, *who* changed it, *when*, and — because you write a message — *why*. You can look at any past snapshot, compare two of them, or restore one.

For science this is not a convenience, it is part of **reproducibility**: the code that produced a result should be recoverable, exactly, forever. Journals and funding agencies increasingly expect analysis code to be public, and "public" in practice means "on GitHub".

### B2. git is not GitHub

This is the most common confusion, so let's be explicit:

| | **git** | **GitHub** |
|---|---|---|
| What it is | A command-line *program* | A *website* / company (owned by Microsoft) |
| Where it runs | On your computer (here: your CoCalc project) | On GitHub's servers |
| What it does | Tracks changes to files in a folder | Hosts copies of git repositories online, adds a web interface, issues, collaboration tools |
| Does it need the other? | **No.** git works fully offline. | Yes — GitHub is built around git. |

Alternatives to GitHub exist (GitLab, Bitbucket, Codeberg). They all speak git. Everything you learn about git transfers.

### B3. The four places a change can be

A git project (a **repository**, or **repo**) is just a folder with a hidden `.git/` subfolder where git keeps its records. A change you make moves through these places:

```
 ┌──────────────────┐   git add    ┌───────────────┐  git commit  ┌──────────────┐   git push   ┌──────────────┐
 │ Working directory│ ───────────▶ │ Staging area  │ ───────────▶ │  Local repo  │ ───────────▶ │ Remote repo  │
 │ (the files you   │              │ (what will go │              │ (.git/ — all │              │  (GitHub)    │
 │  actually edit)  │ ◀─────────── │  in the next  │              │  commits so  │ ◀─────────── │              │
 └──────────────────┘ git restore  │  commit)      │              │  far)        │ git fetch /  └──────────────┘
                                   └───────────────┘              └──────────────┘   git pull

```

* **Working directory** — the ordinary files you edit with any editor.
* **Staging area** (also called the *index*) — a holding pen. `git add` puts a file's *current* state here. This lets you commit some changes but not others.
* **Local repository** — `git commit` takes everything in the staging area and records it permanently as a commit, with a message. This lives in `.git/` inside your project.
* **Remote repository** — a copy on GitHub. `git push` sends your new commits up; `git fetch` / `git pull` bring down commits that others (or you, from another machine) pushed.

The thing students most often get wrong: **editing a file does not change the repository.** Neither does `git add` on its own. Only `git commit` records history, and only `git push` sends it to GitHub.

### B4. The everyday workflow

Ninety percent of your git life is this loop:

1. Edit files.
2. `git status` — what has changed?
3. `git add <file>` — stage the changes you want to record (or `git add .` for everything).
4. `git commit -m "Short message describing why"`
5. `git push` — send to GitHub.

Plus, when starting: `git clone <url>` to get a copy of an existing repo, and `git pull` to bring in changes others have pushed. `git pull` is really two steps — `git fetch` (download the remote's commits, without touching your files) followed by a merge into your working copy. Running `git fetch` on its own, then `git status`, lets you see whether you are ahead of or behind the remote before you change anything.

**Commit messages.** Write them for the person reading `git log` in a year (probably you). `Fix off-by-one in GC window` is good. `update`, `asdf`, `final fix` are useless. Convention: short imperative sentence, under ~70 characters.

### B5. Command cheat sheet

| Command | What it does |
|---|---|
| `git clone <url>` | Copy a remote repo (and its full history) onto your machine |
| `git init` | Turn the current folder into a new, empty repo |
| `git status` | Show which files are modified / staged / untracked. **Run this constantly.** |
| `git add <file>` | Stage a file. `git add .` stages everything in the folder |
| `git commit -m "msg"` | Record staged changes as a commit |
| `git log` | List commits. `git log --oneline` is more compact |
| `git diff` | Show unstaged changes. `git diff --staged` shows staged ones |
| `git push` | Send local commits to the remote |
| `git fetch` | Download commits from the remote without changing your files; then `git status` shows whether you are ahead or behind |
| `git pull` | `git fetch` + merge the remote's commits into your working copy |
| `git remote -v` | Show which remote(s) this repo is connected to |
| `git restore <file>` | Discard unstaged edits to a file (careful — no undo) |
| `git stash` / `git stash pop` | Shelve all uncommitted changes (e.g. so you can `git pull` cleanly), then bring them back |
| `git checkout <hash> -- <file>` | Bring back a file as it was in an old commit |

You don't need to memorise these; you need to know they exist. `git <command> --help` shows the manual.

### B6. Self-check questions

**Q1.** Your friend says "I committed my changes but they're not showing up on GitHub." Which command did they forget, and why doesn't `git commit` do that job itself?

**Q2.** You edit `analysis.py`, run `git commit -m "fix bug"`, and git replies `no changes added to commit`. What step is missing? What does `git status` show at this moment?

**Q3.** True or false, with one sentence of justification: *If GitHub went offline tomorrow, you would lose the history of a project you have cloned to CoCalc.*

**Q4.** Which of these is the best commit message for a change that corrects how a script reads FASTA headers, and why are the others worse?

(a) `fixed it`  (b) `Changed line 42 of parse.py`  (c) `Handle FASTA headers containing spaces`  (d) `commit`

**Q5.** In your own words, what is the difference between git and GitHub? (Two sentences max.)

## Part C — Dry run: a local repository

Let's make sure you understand git itself before we add GitHub to the mix. Everything in this part stays on CoCalc.

In the terminal, execute the following commands to create a scratch folder and turn it into a repository:

```
$> cd ~
$> mkdir prelab_scratch
$> cd prelab_scratch
$> git init
$> git status
```

`git init` creates the hidden `.git/` folder. `git status` should say `On branch main` and `No commits yet`.

Now create a file. In the terminal, execute:

```
$> echo "ATGCGCGATATCGCGC" > seq.txt
$> git status
```

`seq.txt` is listed as **Untracked** — git sees it but is not recording it. In the terminal, stage it and commit:

```
$> git add seq.txt
$> git status
$> git commit -m "Add example sequence"
$> git log
```

Notice how `git status` changed after `git add` (the file is now under *Changes to be committed*), and how `git log` shows your name, email, timestamp and message.

Now change the file and make a second commit. In the terminal, execute:

```
$> echo "GGCCAATT" >> seq.txt
$> git status
$> git diff
$> git add seq.txt
$> git commit -m "Append second fragment"
$> git log --oneline
```

`git diff` showed you exactly which line was added before you committed it. Run the verification cell:

In [ ]:
# Verification: you should see exactly two commits, newest first.
!cd ~/prelab_scratch && git log --oneline

Finally, make one more change but do **not** add or commit it. In the terminal, execute:

```
$> echo "TTTT" >> seq.txt
$> git status
```

**Q6.** What does `git status` say about `seq.txt` now? Then execute `git diff` — what does it show? Paste or describe the output.

Leave the repo in this state; we will come back to it briefly in class.

## Further reading

* *git - the simple guide* — <https://rogerdudler.github.io/git-guide/> — a one-page, no-nonsense walkthrough of the same commands; good to keep open in a tab.
* The Software Carpentry lesson *Version Control with Git* — <https://swcarpentry.github.io/git-novice/> — covers the same material at a gentler pace.
* *Pro Git* (free book) — <https://git-scm.com/book/en/v2> — chapters 1–2 for the full picture.
* GitHub's docs on SSH — <https://docs.github.com/en/authentication/connecting-to-github-with-ssh> — if Part A gave you trouble.